# Semantic mode slots on DeMo (Kaggle GPU)
Kaggle version of the Colab notebook. Free: about 30 GPU hours per week, sessions up to 12 h.

**Before you start (once)**
- Kaggle account with a **verified phone number** (needed for internet and GPU).
- Right panel → **Session options**: Accelerator **GPU T4 x2**, **Internet on**.

**How to run**
- No restart is needed. Run the cells from top to bottom.
- Everything below the setup runs as `!python`, so packages installed in K1 are picked up.
- Big data goes to `/tmp` and is lost when the session ends. Checkpoints and results go to `/kaggle/working`.
- Results to send back are marked **Send**.

## K1. Code, packages, checkpoint (about 3 min)

In [ ]:
import os, sys, shutil, subprocess
print(sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | cat
WORK = '/kaggle/working'; DATA = '/tmp'
os.chdir(WORK)
REPO, BRANCH = 'https://github.com/Nohahamid/DeMo.git', 'named-intentions'
if not os.path.exists('DeMo'):
    !git clone -q -b {BRANCH} {REPO} DeMo
os.chdir(f'{WORK}/DeMo')
!git pull -q origin {BRANCH}; git log -1 --format='code version: %h %s'
!pip install -q av2==0.2.1 hydra-core==1.3.2 pytorch-lightning torchmetrics==0.10.2 timm einops rich gdown tensorboard "numpy<2" ninja packaging wheel
!pip uninstall -y -q mamba_ssm causal_conv1d
!cd third_party/mamba && MAMBA_SKIP_CUDA_BUILD=TRUE MAMBA_FORCE_BUILD=TRUE pip install -q . --no-build-isolation --no-deps --no-cache-dir
os.makedirs('ckpts', exist_ok=True)
if not os.path.exists('ckpts/DeMo.ckpt'):
    !gdown -q 1xqj8T5M2cczIZU26poseAQri6VE6v9a0 -O ckpts/DeMo.ckpt
!ls -lh ckpts
!python -c "import torch; from mamba_ssm.modules.mamba_simple import Mamba; m=Mamba(d_model=128,bimamba=True).cuda(); print('Mamba OK', m(torch.randn(2,60,128,device='cuda')).shape, 'GPUs:', torch.cuda.device_count())"

### K2. Download helper
Same as in Colab, and it picks the same scenarios.

In [ ]:
import socket, time, re, urllib.request, urllib.parse, concurrent.futures, pathlib
from tqdm.auto import tqdm
socket.setdefaulttimeout(30)
def download(root, split, n):
    base = 'https://argoverse.s3.us-east-1.amazonaws.com'
    keys, token = [], None
    while len(keys) < 2 * n:
        url = f'{base}/?list-type=2&prefix=datasets/av2/motion-forecasting/{split}/&max-keys=1000'
        if token: url += '&continuation-token=' + urllib.parse.quote(token)
        xml = urllib.request.urlopen(url).read().decode()
        keys += re.findall(r'<Key>([^<]+)</Key>', xml)
        m = re.search(r'<NextContinuationToken>([^<]+)</NextContinuationToken>', xml)
        if not m: break
        token = m.group(1)
    keys = keys[:2 * n]
    def fetch(k):
        out = pathlib.Path(root) / split / pathlib.Path(k).parent.name / pathlib.Path(k).name
        if out.exists(): return
        out.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(3):
            try:
                tmp = out.with_suffix(out.suffix + '.tmp')
                urllib.request.urlretrieve(f'{base}/{k}', tmp); tmp.rename(out); return
            except Exception:
                time.sleep(2)
    with concurrent.futures.ThreadPoolExecutor(32) as ex: list(tqdm(ex.map(fetch, keys), total=len(keys)))
    print(len(list(pathlib.Path(root, split).iterdir())), split, 'scenarios in', root)

## K3. 5,000 val scenarios: download, preprocess, labels, slots

In [ ]:
download(f'{DATA}/av2_val5k', 'val', 5000)
os.makedirs(f'{DATA}/val5k', exist_ok=True)
!cd {DATA}/val5k && python {WORK}/DeMo/preprocess.py --data_root {DATA}/av2_val5k -p
!ls {DATA}/val5k/data/DeMo_processed/val | wc -l
!python intention_labels.py --data_root {DATA}/av2_val5k --split val --out data/intention_labels_sem/val.csv
!python semantic_slots.py --data_root {DATA}/av2_val5k --split val --out data/semantic_slots/val.csv

## K4. 50k training scenarios: download, preprocess, labels, slots (the long one)
At the end, **Send** the `df -h` line and the counts.

In [ ]:
download(f'{DATA}/av2_raw', 'train', 50000)
!cd {DATA} && python {WORK}/DeMo/preprocess.py --data_root {DATA}/av2_raw -p
!ls {DATA}/data/DeMo_processed/train | wc -l
!python intention_labels.py --data_root {DATA}/av2_raw --split train --out data/intention_labels_sem/train.csv
!python semantic_slots.py --data_root {DATA}/av2_raw --split train --out data/semantic_slots/train.csv
!rm -rf {DATA}/data/DeMo_processed/val
!ln -s {DATA}/val5k/data/DeMo_processed/val {DATA}/data/DeMo_processed/val
!ls {DATA}/data/DeMo_processed/val | wc -l
!df -h /tmp | cat

## K5. Full training (H5b settings from the smoke tests)
- Two T4s, 16 scenarios each, so the batch is still 32.
- If it fails with a DDP or "unused parameters" error, change to `gpus=1 batch_size=32`.
- **Send** the speed (`it/s`) after a few minutes, and the last Validate table at the end.

In [ ]:
!python train.py datamodule=av2 model=model_forecast gpus=2 sync_bn=false \
    batch_size=16 num_workers=4 \
    datamodule.target.data_root={DATA}/data/DeMo_processed \
    pretrained_weights=ckpts/DeMo.ckpt num_intentions=0 \
    semantic_slots=true slot_dir=data/semantic_slots \
    intention_dir=data/intention_labels_sem train_scope=decoder \
    "oversample={1:2,2:2,3:3,4:3}" semantic_weight=0.1 \
    epochs=3 warmup_epochs=0 lr=0.00005

## K6. Diagnosis of the new model
**Send** the whole output. Then use the right panel (Output) to download `sem_val.npz` and `last.ckpt`, and keep them on your laptop or Drive.

In [ ]:
import glob
CK = max(glob.glob('outputs/**/checkpoints/last.ckpt', recursive=True), key=os.path.getmtime)
print(CK)
!python fusion_intent.py extract --ckpt {CK} --split val \
    --data_root {DATA}/val5k/data/DeMo_processed --raw_root {DATA}/av2_val5k \
    --intention_dir data/intention_labels_sem --slot_dir data/semantic_slots --out {WORK}/sem_val.npz
!python fusion_intent.py diagnose --val {WORK}/sem_val.npz